In [1]:
import json
from pathlib import Path
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, AddedToken
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from huggingface_hub import hf_hub_download

In [2]:
VOCAB_SIZE = 16_000
MIN_FREQUENCY = 2
BATCH_SIZE = 1_000

OUTPUT_DIR = Path("tokenizer")

In [3]:
SPECIAL_TOKENS = [
    "<|beginoftext|>",
    "<|endoftext|>",
    "<|unk|>",
]

In [4]:
path = hf_hub_download("roneneldan/TinyStories",
                       "TinyStoriesV2-GPT4-train.txt",
                       repo_type="dataset")

In [5]:
def text_iterator(path, batch_size=BATCH_SIZE):
    """Yield batches of stories without loading the full file."""
    batch = []

    with Path(path).open("r", encoding="utf-8") as file:
        for line in file:
            text = line.strip()

            if not text:
                continue

            batch.append(text)

            if len(batch) >= batch_size:
                yield batch
                batch = []

    if batch:
        yield batch


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [7]:
tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))

In [8]:
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

In [9]:
tokenizer.decoder = ByteLevelDecoder()

In [10]:
trainer = BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=MIN_FREQUENCY,
        special_tokens=[
            AddedToken(token, special=True)
            for token in SPECIAL_TOKENS
        ],
        initial_alphabet=ByteLevel.alphabet(),
        show_progress=True,
    )

In [11]:
tokenizer.train_from_iterator(
        text_iterator(path),
        trainer=trainer,
    )

In [12]:
tokenizer_path = OUTPUT_DIR / "tokenizer.json"
tokenizer.save(str(tokenizer_path))

In [13]:
token_ids = {
        token: tokenizer.token_to_id(token)
        for token in SPECIAL_TOKENS
    }


In [14]:
with (OUTPUT_DIR / "special_tokens.json").open("w", encoding="utf-8") as f:
    json.dump(token_ids, f, indent=2)

In [15]:
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Tokenizer saved to:", tokenizer_path)
print("Special-token IDs:", token_ids)

Vocabulary size: 16000
Tokenizer saved to: tokenizer/tokenizer.json
Special-token IDs: {'<|beginoftext|>': 0, '<|endoftext|>': 1, '<|unk|>': 2}


In [16]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))

In [17]:
tests = [
    "Once upon a time, there was a little fox.",
    "The little girl went to the forest.",
    "He was happy!",
    "The dragon could fly over the mountains.",
]


In [18]:
for text in tests:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)

    assert decoded == text, (
        f"Round-trip failed: {text!r} -> {decoded!r}"
    )

    print("Text:", text)
    print("Tokens:", encoded.tokens)
    print("IDs:", encoded.ids)
    print("Decoded:", decoded)
    print("-" * 40)

Text: Once upon a time, there was a little fox.
Tokens: ['Once', 'Ġupon', 'Ġa', 'Ġtime', ',', 'Ġthere', 'Ġwas', 'Ġa', 'Ġlittle', 'Ġfox', '.']
IDs: [439, 448, 261, 405, 14, 409, 285, 261, 399, 1686, 16]
Decoded: Once upon a time, there was a little fox.
----------------------------------------
Text: The little girl went to the forest.
Tokens: ['The', 'Ġlittle', 'Ġgirl', 'Ġwent', 'Ġto', 'Ġthe', 'Ġforest', '.']
IDs: [419, 399, 487, 436, 268, 265, 976, 16]
Decoded: The little girl went to the forest.
----------------------------------------
Text: He was happy!
Tokens: ['He', 'Ġwas', 'Ġhappy', '!']
IDs: [1034, 285, 385, 3]
Decoded: He was happy!
----------------------------------------
Text: The dragon could fly over the mountains.
Tokens: ['The', 'Ġdragon', 'Ġcould', 'Ġfly', 'Ġover', 'Ġthe', 'Ġmountains', '.']
IDs: [419, 1882, 476, 850, 887, 265, 6103, 16]
Decoded: The dragon could fly over the mountains.
----------------------------------------


In [19]:
for token in [
    "<|beginoftext|>",
    "<|endoftext|>",
    "<|unk|>",
]:
    assert tokenizer.token_to_id(token) is not None

In [46]:
batch = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        text = line.strip()
        print("Text:", text)
        batch.append(text)
        if len(batch)==10:
            break
        if not text:
            continue

        encoded = tokenizer.encode(text)
        decoded = tokenizer.decode(encoded.ids, skip_special_tokens=False)
        
        assert decoded == text, (
            f"Round-trip failed: {text!r} -> {decoded!r}"
        )

Text: 
Text: Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!
Text: He said, “Wow, that is a really amazing vase! Can I buy it?”
Text: The shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”
Text: So Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was.
Text: And that's how Ben found an amazing vase in the store!
Text: <|endoftext|>
Text: Once upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.
Text: One day, Ollie's mom said, "Ollie, hurry and get some fish for dinn